In [ ]:
import re
import pandas as pd
from collections import Counter
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [ ]:
import matplotlib.colors as mcolors
from matplotlib.colorbar import ColorbarBase
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

rcParams['figure.dpi'] = 500
rcParams['savefig.dpi'] = 500
rcParams['font.family'] = 'Arial'
rcParams['axes.labelsize'] = 14
rcParams['axes.titlesize'] = 14
rcParams['xtick.labelsize'] = 14
rcParams['ytick.labelsize'] = 14
rcParams['legend.fontsize'] = 14
rcParams['figure.titlesize'] = 16

In [ ]:
mmi_map = {
    'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6,
    'VII': 7, 'VIII': 8, 'IX': 9, 'X': 10, 'XI': 11, 'XII': 12
}

mmi_colors = {
        'I': '#B9D3F6',    # Light blue
        'II': '#9FC5F4',
        'III': '#86B7F2',  # Blue
        'IV': '#70E7F7',   # Cyan
        'V': '#92E285',    # Green
        'VI': '#FFFF00',   # Yellow
        'VII': '#FFC800',  # Orange
        'VIII': '#FF9100', # Dark orange
        'IX': '#FF0000',   # Red
        'X': '#C80000',    # Dark red
        'XI': '#A00000',   # Darker red
        'XII': '#800000'   # Very dark red
    }

In [ ]:
def extract_id(path):
    match = re.search(r'/(\d+_\d+)_', path)
    return match.group(1) if match else None


def process_dataframe(filepath):
    df = pd.read_json(filepath)
    df['location_id'] = df['file_path'].apply(extract_id)
    prompt_df = pd.read_csv('2019_ridgecrest_samples_prompt.csv')

    df = pd.merge(prompt_df, df, on='location_id', how='inner')
    df['MMI_predicted_num'] = df['MMI_predicted'].map(mmi_map)

    for col in df.columns:
        if col.endswith('_x') and col[:-2] + '_y' in df.columns:
            if df[col].equals(df[col[:-2] + '_y']):
                df.drop(columns=[col[:-2] + '_y'], inplace=True)
                df.rename(columns={col: col[:-2]}, inplace=True)
            else:
                df.drop(columns=[col, col[:-2] + '_y'], inplace=True)
                
    return df

df1 = process_dataframe('result_zipcode/2019_ridgecrest_B+G+B+C+V_gpt-4.1-mini-2025-04-14.json')
df2 = process_dataframe('result_zipcode/2019_ridgecrest_B+G+B+C+V_Qwen2.5-VL-32B-Instruct.json')

In [ ]:
def plot_mmi_relationship(df, x_col='MMI_predicted', y_col='distance', line_color='red'):
    # Map Roman to numeric MMI
    df['MMI_predicted_num'] = df[x_col].map(mmi_map)
    df[y_col] = pd.to_numeric(df[y_col], errors='coerce')
    df_clean = df.dropna(subset=['MMI_predicted_num', y_col])
    
    # Add color column for plotting
    df_clean['color'] = df_clean[x_col].map(mmi_colors)
    
    # Add jitter to x-values
    jitter_amount = 0.2
    df_clean['x_jittered'] = df_clean['MMI_predicted_num'] + np.random.uniform(
        -jitter_amount, jitter_amount, size=len(df_clean)
    )
    
    # Linear regression
    X = df_clean['MMI_predicted_num'].values.reshape(-1, 1)
    y = df_clean[y_col].values
    reg = LinearRegression().fit(X, y)
    y_pred = reg.predict(X)
    coef = reg.coef_[0]
    r2 = r2_score(y, y_pred)
    
    # Create joint plot with smaller marginal plots
    g = sns.JointGrid(
        data=df_clean, 
        x='x_jittered', 
        y=y_col,
        height=8
    )
    
    g.fig.set_size_inches((3.5, 3.6))
    
    # Add colored scatter plot to the main plot
    for mmi in sorted(df_clean[x_col].unique()):
        mask = df_clean[x_col] == mmi
        mmi_num = mmi_map[mmi]
        color = mmi_colors[mmi]
        
        g.ax_joint.scatter(
            df_clean.loc[mask, 'x_jittered'],
            df_clean.loc[mask, y_col],
            color=color,
            alpha=0.7,
            s=50,
            edgecolor='none'
        )
    
    # Add regression line
    x_range = np.array([min(df_clean['MMI_predicted_num']), max(df_clean['MMI_predicted_num'])]).reshape(-1, 1)
    g.ax_joint.plot(
        x_range.flatten(),
        reg.predict(x_range),
        color=line_color,
        linestyle='--',
        linewidth=2
    )
    
    # Add marginal plots
    # X-axis marginal: colored histogram by MMI category
    for mmi in sorted(df_clean[x_col].unique()):
        mask = df_clean[x_col] == mmi
        mmi_num = mmi_map[mmi]
        color = mmi_colors[mmi]
        
        g.ax_marg_x.bar(
            mmi_num, 
            sum(mask), 
            width=0.8, 
            color=color,
            alpha=0.8,
            edgecolor='none'
        )
    
    # Y-axis marginal: KDE plot for distance
    sns.kdeplot(
        y=df_clean[y_col], 
        ax=g.ax_marg_y, 
        fill=True, 
        color='#86B7F2', 
        alpha=0.6
    )
    
    # Label formatting
    g.ax_joint.set_xlabel("Predicted MMI", fontsize=12)
    
    if y_col == "distance":
        g.ax_joint.set_ylabel("Distance (km)", fontsize=12)
    elif y_col == "vs30":
        g.ax_joint.set_ylabel("VS30 (m/s)", fontsize=12)
    
    # Replace X ticks with Roman numerals
    ticks = sorted(df_clean['MMI_predicted_num'].unique())
    roman_ticks = [k for k, v in mmi_map.items() if v in ticks]
    g.ax_joint.set_xticks(ticks)
    g.ax_joint.set_xticklabels(roman_ticks, fontsize=12)
    
    # Annotate regression equation
    eqn_text = (
        rf"$y = {coef:.2f}x + {reg.intercept_:.2f}$" + "\n"
        rf"$R^2 = {r2:.3f}$"
    )
    
    g.ax_joint.annotate(
        eqn_text,
        xy=(0.98, 0.98), xycoords='axes fraction',
        ha='right', va='top',
        fontsize=12, color=line_color,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none", alpha=0.8)
    )
    
    # Remove marginal labels and clean up
    g.ax_marg_x.set_yticks([])
    g.ax_marg_x.set_ylabel('')
    g.ax_marg_y.set_xticks([])
    g.ax_marg_y.set_xlabel('')
    
    # Clean up and remove spines
    for ax in [g.ax_joint, g.ax_marg_x, g.ax_marg_y]:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(False)
    
    plt.tight_layout()

In [ ]:
plot_mmi_relationship(df1, x_col='MMI_predicted', y_col='distance', line_color='#C80000')
plot_mmi_relationship(df1, x_col='MMI_predicted', y_col='vs30', line_color='#C80000')

In [ ]:
plot_mmi_relationship(df2, x_col='MMI_predicted', y_col='distance', line_color='#C80000')
plot_mmi_relationship(df2, x_col='MMI_predicted', y_col='vs30', line_color='#C80000')